# WIMP Demo - Worm IMage Processor

This notebook demonstrates the core capabilities of the **WIMP** package for
time-domain NV-center magnetometry signal processing applied to *C. elegans*
neural imaging.

Topics covered:
1. Synthetic data generation
2. Curve fitting (Ramsey, Echo, T1)
3. Sensitivity comparison across protocols
4. Field time-series & neural source localisation
5. Neural event detection
6. Pipeline processing
7. Real-time streaming processing
8. Deformable atlas registration
9. Crosstalk analysis
10. Data I/O

In [ ]:
%pip install -q -e .

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import wimp
from wimp.synthetic import (
    generate_ramsey_data,
    generate_echo_data,
    generate_t1_data,
    generate_full_experiment,
)
from wimp.relaxation import fit_ramsey, fit_t2_decay, fit_t1_recovery
from wimp.sensitivity import compare_protocols, optimal_protocol
from wimp.analysis import psd_welch, detect_neural_events
from wimp import viz

print(f"WIMP version: {wimp.__version__}")

## 1. Synthetic Ramsey Data & Fitting

Generate a synthetic Ramsey fringe with known ground-truth parameters,
then recover them via non-linear least-squares fitting.

In [ ]:
# Ground truth
B_TRUE = 50e-6      # 50 uT
T2STAR_TRUE = 1e-6  # 1 us

tau = np.linspace(0, 5e-6, 200)
data = generate_ramsey_data(tau, b_field=B_TRUE, t2star=T2STAR_TRUE, snr=40, seed=42)

# Fit
fit = fit_ramsey(data["tau"], data["signal"])

print(f"True B-field:  {B_TRUE*1e6:.1f} uT")
print(f"Fitted B-field: {fit['b_field']*1e6:.1f} uT")
print(f"True T2*:      {T2STAR_TRUE*1e6:.2f} us")
print(f"Fitted T2*:    {fit['t2star']*1e6:.2f} us")
print(f"R-squared:     {fit['r_squared']:.4f}")

In [ ]:
fig = viz.plot_ramsey_fringe(data["tau"], data["signal"], fit)
plt.show()

## 2. Echo & T1 Fitting

In [ ]:
# Hahn echo
tau_echo = np.linspace(0, 300e-6, 150)
echo_data = generate_echo_data(tau_echo, t2=100e-6, snr=50, seed=7)
echo_fit = fit_t2_decay(echo_data["tau"], echo_data["signal"])

print(f"True T2:   {100:.0f} us")
print(f"Fitted T2: {echo_fit['t2']*1e6:.1f} us")

fig = viz.plot_decay_curve(echo_data["tau"], echo_data["signal"], echo_fit)
plt.show()

In [ ]:
# T1 recovery
tau_t1 = np.linspace(0, 25e-3, 150)
t1_data = generate_t1_data(tau_t1, t1=5e-3, snr=50, seed=12)
t1_fit = fit_t1_recovery(t1_data["tau"], t1_data["signal"])

print(f"True T1:   {5:.1f} ms")
print(f"Fitted T1: {t1_fit['t1']*1e3:.2f} ms")

fig = viz.plot_t1_recovery(t1_data["tau"], t1_data["signal"], t1_fit)
plt.show()

## 3. Protocol Sensitivity Comparison

Compare the magnetic-field sensitivity achievable with different
NV magnetometry protocols.

In [ ]:
df = compare_protocols(contrast=0.03, t2star=1e-6, t2=100e-6, t1=5e-3)
display(df)

rec = optimal_protocol("ac")
print(f"\nRecommended AC protocol: {rec['protocol']}")
print(f"Sensitivity: {rec['sensitivity_T_sqrtHz']*1e9:.2f} nT/sqrt(Hz)")

In [ ]:
fig = viz.plot_sensitivity_comparison(df)
plt.show()

## 4. Full Synthetic Experiment & Source Localisation

Generate a complete multi-ND experiment with simulated neural sources,
then reconstruct source currents via the MNE inverse solver.

In [ ]:
exp = generate_full_experiment(
    n_nds=8, n_neurons=4, n_timepoints=500,
    protocol="ramsey", snr=30, seed=99,
)

print(f"Signal shape:       {exp['signal'].shape}")
print(f"Field timeseries:   {exp['field_timeseries'].shape}")
print(f"ND positions:       {exp['nd_positions'].shape}")
print(f"Neuron positions:   {exp['neuron_positions'].shape}")

In [ ]:
# Plot field time-series at each ND
fig = viz.plot_field_timeseries(exp["time"], exp["field_timeseries"])
plt.show()

In [ ]:
# Source localisation using MNE inverse
from wimp.source import lead_field_matrix, mne_inverse

L = lead_field_matrix(exp["neuron_positions"], exp["nd_positions"])
source_est = mne_inverse(L, exp["field_timeseries"], lambda_reg=0.1)

print(f"Source estimate shape: {source_est.shape}")
print(f"  (n_neurons x n_timepoints)")

In [ ]:
# Compare estimated vs true source activity
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

t_ms = exp["time"] * 1e3
for j in range(exp["current_waveforms"].shape[0]):
    axes[0].plot(t_ms, exp["current_waveforms"][j], lw=0.8, label=f"Neuron {j}")
axes[0].set_ylabel("True current (a.u.)")
axes[0].set_title("Ground-truth neural activity")
axes[0].legend(fontsize=7)

for j in range(source_est.shape[0]):
    axes[1].plot(t_ms, source_est[j], lw=0.8, label=f"Neuron {j}")
axes[1].set_ylabel("Estimated current (a.u.)")
axes[1].set_xlabel("Time (ms)")
axes[1].set_title("MNE source reconstruction")
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# Source map and field map
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

mean_src = np.mean(np.abs(source_est), axis=1)
viz.plot_source_map(exp["neuron_positions"], mean_src, ax=axes[0])

mean_field = np.mean(np.abs(exp["field_timeseries"]), axis=1)
viz.plot_field_map_2d(exp["nd_positions"], mean_field, ax=axes[1])

plt.tight_layout()
plt.show()

## 5. Neural Event Detection

In [ ]:
fs = 1.0 / exp["dt"]
events = detect_neural_events(exp["field_timeseries"], fs, threshold_sigma=2.5)

print(f"Detected {len(events['event_times'])} neural events")
print(f"Event times (ms): {np.round(events['event_times']*1e3, 1)[:10]}...")

## 6. Pipeline Processing

Run the full WIMP pipeline on the synthetic data via the high-level API.

In [ ]:
from wimp.io import WIMPDataset
from wimp.pipeline import PipelineConfig, run_pipeline

ds = WIMPDataset(
    protocol="ramsey",
    tau_array=exp["tau_array"],
    signal=exp["signal"],
    nd_positions=exp["nd_positions"],
)

config = PipelineConfig(
    protocol="ramsey",
    source_localization=True,
    lambda_reg=0.1,
    registration_params={"neuron_positions": exp["neuron_positions"].tolist()},
)

results = run_pipeline(config, dataset=ds)

print(f"Processed {results['n_nds']} NDs")
print(f"Source localisation: {'yes' if results['source_estimate'] is not None else 'no'}")

## 7. Real-time Streaming Demo

Demonstrate the real-time processor by streaming synthetic frames.

In [ ]:
import time
from wimp.realtime import RealtimeProcessor, RealtimeConfig

rt_config = RealtimeConfig(
    protocol="ramsey",
    buffer_size=50,
    fit_interval=0.1,
    averaging=3,
)

# Collect results in a list
rt_results = []

def on_result(result):
    rt_results.append(result)

proc = RealtimeProcessor(rt_config)
proc.on_result(on_result)
proc.start()

# Simulate streaming 20 Ramsey acquisitions
tau_rt = np.linspace(0, 5e-6, 100)
for i in range(20):
    frame_data = generate_ramsey_data(
        tau_rt, b_field=50e-6, t2star=1e-6, snr=20,
    )
    proc.push_arrays(tau_rt, frame_data["signal"])
    time.sleep(0.05)  # simulate 50 ms acquisition interval

time.sleep(0.5)  # let background thread catch up
proc.stop()

print(f"Frames pushed: {proc.frame_count}")
print(f"Fit results received: {len(rt_results)}")

if rt_results:
    last = rt_results[-1]
    print(f"Latest B-field estimate: {last.field_values[0]*1e6:.1f} uT")

In [ ]:
# Plot the evolution of field estimates over time
if rt_results:
    b_vals = [r.field_values[0] * 1e6 for r in rt_results]  # uT
    frames = [r.frame_count for r in rt_results]

    plt.figure(figsize=(8, 3))
    plt.plot(frames, b_vals, "o-", ms=4)
    plt.axhline(50, color="r", ls="--", lw=1, label="True value (50 uT)")
    plt.xlabel("Frame count")
    plt.ylabel("Estimated B (uT)")
    plt.title("Real-time field tracking")
    plt.legend()
    plt.tight_layout()
    plt.show()

## 8. Deformable Atlas Registration

Run per-frame deformable atlas registration on a simulated moving worm,
then visualise neuron trajectories over time.

In [ ]:
from wimp.synthetic import generate_deformable_experiment
from wimp.registration import deformable_register, smooth_neuron_trajectories
from wimp.viz import plot_neuron_trajectories, plot_deformable_registration

# Generate deformable worm experiment
deform_exp = generate_deformable_experiment(
    n_nds=10, n_neurons=5, n_frames=20,
    amplitude=10e-6, seed=42,
)
print(f"Frame ND positions:     {deform_exp['frame_nd_positions'].shape}")
print(f"Frame neuron positions: {deform_exp['frame_neuron_positions'].shape}")
print(f"Field timeseries:       {deform_exp['field_timeseries'].shape}")

# Run deformable registration
dereg = deformable_register(deform_exp["frame_nd_positions"])
print(f"\nRegistered {len(dereg.neuron_names)} neurons across {len(dereg.centerlines)} frames")

# Smooth trajectories
smoothed = smooth_neuron_trajectories(dereg.neuron_positions, window=3)
print(f"Smoothed positions shape: {smoothed.shape}")

In [ ]:
# Visualise deformable registration at selected frames
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for idx, frame in enumerate([0, 9, 19]):
    nd_pos_frame = deform_exp["frame_nd_positions"][frame]
    plot_deformable_registration(dereg, frame, nd_pos_frame, ax=axes[idx])
    axes[idx].set_title(f"Frame {frame}")
plt.tight_layout()
plt.show()

# Plot neuron trajectories over time
fig = plot_neuron_trajectories(
    dereg.neuron_positions[:, :5],
    dereg.neuron_names[:5],
)
plt.show()

## 9. Crosstalk Analysis

Compute the resolution matrix **R = K L** to quantify how much signal
from each neuron leaks into the estimates of other neurons.  A perfect
inverse would give R = I (identity).

In [ ]:
from wimp.source import resolution_matrix, crosstalk_metrics
from wimp.viz import plot_resolution_matrix, plot_crosstalk_summary

# Reuse the lead-field matrix L from Section 4
R = resolution_matrix(L, lambda_reg=0.1)
metrics = crosstalk_metrics(R)

print("Resolution matrix R (should be close to identity):")
print(np.round(R, 3))
print(f"\nSelf-resolution (diagonal): {np.round(metrics['diagonal'], 3)}")
print(f"Crosstalk ratio:           {np.round(metrics['crosstalk_ratio'], 3)}")
print(f"Total leakage:             {np.round(metrics['total_leakage'], 3)}")

# Visualise resolution matrix heatmap
fig = plot_resolution_matrix(R, metrics["neuron_names"])
plt.show()

# Crosstalk summary bar chart
fig = plot_crosstalk_summary(metrics)
plt.show()

In [ ]:
# Compare crosstalk across different regularisation strengths
lambdas = [0.01, 0.1, 0.5, 1.0]
fig, axes = plt.subplots(1, len(lambdas), figsize=(14, 3.5))

for ax, lam in zip(axes, lambdas):
    R_lam = resolution_matrix(L, lambda_reg=lam)
    plot_resolution_matrix(R_lam, ax=ax, vmax=1.0)
    diag_mean = np.diag(R_lam).mean()
    ax.set_title(f"$\\lambda$={lam}\ndiag={diag_mean:.2f}")

plt.suptitle("Effect of regularisation on the resolution matrix", y=1.02)
plt.tight_layout()
plt.show()

## 10. Data I/O

Save and reload datasets in HDF5 and NumPy formats.

In [ ]:
import tempfile, os
from wimp.io import save_hdf5, load_hdf5, save_numpy, load_numpy

with tempfile.TemporaryDirectory() as tmpdir:
    # HDF5 round-trip
    h5_path = os.path.join(tmpdir, "test.h5")
    save_hdf5(ds, h5_path)
    ds_loaded = load_hdf5(h5_path)
    print(f"HDF5 round-trip: signal shape = {ds_loaded.signal.shape}")
    
    # NumPy round-trip
    npz_dir = os.path.join(tmpdir, "npz_out")
    save_numpy(ds, npz_dir)
    ds_loaded2 = load_numpy(npz_dir)
    print(f"NPZ round-trip:  signal shape = {ds_loaded2.signal.shape}")
    
    print("\nAll I/O round-trips successful!")

---
*WIMP - Worm IMage Processor | iGEM 2026 Team UIUC*